In [27]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from config.settings import *

import pandas as pd
import geopandas as gpd
import numpy as np
import fiona

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)

Project root: /Users/yongyili/EnvHealthLondon
Data dir: /Users/yongyili/EnvHealthLondon/data


In [28]:
os.listdir(DATA_DIR)

['laqn_borough.parquet',
 'London_Borough_Excluding_MHW.shp',
 'London_Borough_Excluding_MHW.shx',
 'laqn_raw.parquet',
 '.DS_Store',
 'greenspace_borough.parquet',
 'noise_borough.parquet',
 'opgrsp_gb.gpkg',
 'London_Borough_Excluding_MHW.dbf',
 'London_Borough_Excluding_MHW.prj',
 'imd_borough.parquet',
 'master_borough.parquet',
 'Road_Noise_Lden_England_Round_3.geojson',
 'laqn_clean.parquet',
 'imd_full.csv']

In [30]:
laqn_borough = pd.read_parquet(f"{DATA_DIR}/laqn_borough.parquet")
noise_borough = pd.read_parquet(f"{DATA_DIR}/noise_borough.parquet")
greenspace_borough = pd.read_parquet(f"{DATA_DIR}/greenspace_borough.parquet")
imd_borough = pd.read_parquet(f"{DATA_DIR}/imd_borough.parquet")

print("LAQN:", laqn_borough.shape)
print("Noise:", noise_borough.shape)
print("Greenspace:", greenspace_borough.shape)
print("IMD:", imd_borough.shape)

imd_borough.head()

LAQN: (172, 5)
Noise: (33, 2)
Greenspace: (33, 2)
IMD: (33, 5)


,borough_name,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
8,Barking and Dagenham,32.883473,7244.500000,2.681818,3.636364
9,Barnet,15.953626,19287.450237,6.383886,0.473934
17,Bexley,15.882363,19696.109589,6.506849,0.000000
30,Brent,25.199855,12034.705202,4.161850,5.780347
35,Bromley,13.969447,21707.984772,7.106599,0.507614


In [31]:
imd_borough.sort_values("pct_most_deprived", ascending=False).head(10)

,borough_name,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
112,Hackney,32.868201,7420.611111,2.743056,11.111111
117,Haringey,27.587200,11291.917241,3.903448,9.655172
139,Kensington and Chelsea,22.019301,15419.339806,5.213592,8.737864
30,Brent,25.199855,12034.705202,4.161850,5.780347
94,Enfield,25.403448,12698.519126,4.382514,5.464481
138,Islington,27.706041,10296.512195,3.642276,4.878049
8,Barking and Dagenham,32.883473,7244.500000,2.681818,3.636364
248,Southwark,26.041289,11546.481928,4.024096,3.012048
151,Lewisham,26.869361,10781.378698,3.763314,2.958580
175,Newham,29.771079,8650.018293,3.091463,2.439024


In [4]:
print("LAQN columns:", laqn_borough.columns.tolist())
print("Noise columns:", noise_borough.columns.tolist())
print("Greenspace columns:", greenspace_borough.columns.tolist())
print("IMD columns:", imd_borough.columns.tolist())

LAQN columns: ['borough', 'year', 'no2_mean', 'no2_count', 'site_count']
Noise columns: ['borough_name', 'lden_mean']
Greenspace columns: ['borough_name', 'green_area_m2']
IMD columns: ['borough_name', 'imd_score_mean', 'imd_rank_mean', 'imd_decile_mean', 'pct_most_deprived']


In [5]:
no2_baseline = (
    laqn_borough[laqn_borough["year"].isin([2018, 2019])]
    .groupby("borough")["no2_mean"]
    .mean()
    .reset_index()
    .rename(columns={
        "borough": "borough_name",
        "no2_mean": "no2_baseline_mean"
    })
)

print(no2_baseline.shape)
no2_baseline.head()

(29, 2)


,borough_name,no2_baseline_mean
0,Barking and Dagenham,23.116494
1,Bexley,26.685649
2,Brent,46.512597
3,Bromley,30.871809
4,Camden,53.645008


In [6]:
print("NO2 boroughs:", no2_baseline["borough_name"].nunique())
print("Noise boroughs:", noise_borough["borough_name"].nunique())
print("Greenspace boroughs:", greenspace_borough["borough_name"].nunique())
print("IMD boroughs:", imd_borough["borough_name"].nunique())

NO2 boroughs: 29
Noise boroughs: 33
Greenspace boroughs: 33
IMD boroughs: 33


In [7]:
set(noise_borough["borough_name"]) - set(no2_baseline["borough_name"])

{'Barnet',
 'Hammersmith and Fulham',
 'Hounslow',
 'Kingston upon Thames',
 'Richmond upon Thames',
 'Waltham Forest'}

In [8]:
set(no2_baseline["borough_name"]) - set(noise_borough["borough_name"])

{'Kingston', 'Richmond'}

In [ ]:
# Fix name mismatch

In [9]:
no2_baseline["borough_name"] = no2_baseline["borough_name"].replace({
    "Kingston": "Kingston upon Thames",
    "Richmond": "Richmond upon Thames"
})

In [10]:
print(set(noise_borough["borough_name"]) - set(no2_baseline["borough_name"]))
print(set(no2_baseline["borough_name"]) - set(noise_borough["borough_name"]))

{'Barnet', 'Hammersmith and Fulham', 'Hounslow', 'Waltham Forest'}
set()


In [ ]:
# spatial nearest-station imputation for missing NO₂ boroughs
# “Because LAQN monitoring stations are unevenly distributed, missing borough-level NO₂ baseline values were imputed using nearest-neighbour borough interpolation based on borough centroid distance. These imputed values were flagged as estimated.”

In [11]:
boroughs = gpd.read_file(f"{DATA_DIR}/London_Borough_Excluding_MHW.shp")

boroughs = boroughs[["GSS_CODE", "NAME", "geometry"]].rename(
    columns={
        "GSS_CODE": "borough_code",
        "NAME": "borough_name"
    }
)

boroughs = boroughs.to_crs("EPSG:27700")

In [12]:
no2_full = boroughs[["borough_name", "geometry"]].merge(
    no2_baseline,
    on="borough_name",
    how="left"
)

no2_full["no2_imputed"] = no2_full["no2_baseline_mean"].isna()

print(no2_full[no2_full["no2_imputed"]][["borough_name"]])

              borough_name
3                 Hounslow
9                   Barnet
16          Waltham Forest
22  Hammersmith and Fulham


In [13]:
available = no2_full[~no2_full["no2_baseline_mean"].isna()].copy()
missing = no2_full[no2_full["no2_baseline_mean"].isna()].copy()

available["centroid"] = available.geometry.centroid
missing["centroid"] = missing.geometry.centroid

for idx, row in missing.iterrows():
    distances = available["centroid"].distance(row["centroid"])
    nearest_idx = distances.idxmin()
    nearest_borough = available.loc[nearest_idx, "borough_name"]
    nearest_value = available.loc[nearest_idx, "no2_baseline_mean"]
    
    no2_full.loc[idx, "no2_baseline_mean"] = nearest_value
    no2_full.loc[idx, "no2_source_borough"] = nearest_borough

no2_full["no2_source_borough"] = no2_full["no2_source_borough"].fillna(no2_full["borough_name"])

no2_baseline_full = no2_full[
    ["borough_name", "no2_baseline_mean", "no2_imputed", "no2_source_borough"]
].copy()

no2_baseline_full.head()

,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough
0,Kingston upon Thames,44.309768,False,Kingston upon Thames
1,Croydon,39.499391,False,Croydon
2,Bromley,30.871809,False,Bromley
3,Hounslow,28.230027,True,Richmond upon Thames
4,Ealing,45.988016,False,Ealing


In [14]:
print(no2_baseline_full.shape)
print(no2_baseline_full.isnull().sum())

no2_baseline_full[no2_baseline_full["no2_imputed"]]

(33, 4)
borough_name          0
no2_baseline_mean     0
no2_imputed           0
no2_source_borough    0
dtype: int64


,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough
3,Hounslow,28.230027,True,Richmond upon Thames
9,Barnet,46.512597,True,Brent
16,Waltham Forest,48.655825,True,Hackney
22,Hammersmith and Fulham,28.186041,True,Kensington and Chelsea


In [ ]:
# merge using the full NO₂ table

In [32]:
master = (
    no2_baseline_full
    .merge(noise_borough, on="borough_name", how="outer")
    .merge(greenspace_borough, on="borough_name", how="outer")
    .merge(imd_borough, on="borough_name", how="outer")
)

print(master.shape)
print(master.isnull().sum())
master.sort_values("pct_most_deprived", ascending=False).head(10)

(33, 10)
borough_name          0
no2_baseline_mean     0
no2_imputed           0
no2_source_borough    0
lden_mean             0
green_area_m2         0
imd_score_mean        0
imd_rank_mean         0
imd_decile_mean       0
pct_most_deprived     0
dtype: int64


,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough,lden_mean,green_area_m2,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
11,Hackney,48.655825,False,Hackney,63.962610,3.994272e+06,32.868201,7420.611111,2.743056,11.111111
13,Haringey,28.675485,False,Haringey,64.065944,9.085837e+06,27.587200,11291.917241,3.903448,9.655172
19,Kensington and Chelsea,28.186041,False,Kensington and Chelsea,65.275871,2.515418e+06,22.019301,15419.339806,5.213592,8.737864
3,Brent,46.512597,False,Brent,64.076461,6.242091e+06,25.199855,12034.705202,4.161850,5.780347
9,Enfield,31.360600,False,Enfield,63.013682,1.248222e+07,25.403448,12698.519126,4.382514,5.464481
18,Islington,34.672520,False,Islington,64.512417,1.100483e+06,27.706041,10296.512195,3.642276,4.878049
0,Barking and Dagenham,23.116494,False,Barking and Dagenham,62.939922,6.590169e+06,32.883473,7244.500000,2.681818,3.636364
27,Southwark,35.653926,False,Southwark,64.135170,4.479264e+06,26.041289,11546.481928,4.024096,3.012048
22,Lewisham,37.932047,False,Lewisham,64.038411,6.524244e+06,26.869361,10781.378698,3.763314,2.958580
24,Newham,28.633707,False,Newham,63.416212,6.049061e+06,29.771079,8650.018293,3.091463,2.439024


In [33]:
master.isnull().sum()

borough_name          0
no2_baseline_mean     0
no2_imputed           0
no2_source_borough    0
lden_mean             0
green_area_m2         0
imd_score_mean        0
imd_rank_mean         0
imd_decile_mean       0
pct_most_deprived     0
dtype: int64

In [17]:
print(master.shape)
master.head()

(33, 10)


,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough,lden_mean,green_area_m2,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
0,Barking and Dagenham,23.116494,False,Barking and Dagenham,62.939922,6.590169e+06,32.883473,7244.500000,2.681818,3.636364
1,Barnet,46.512597,True,Brent,63.562162,1.748205e+07,15.953626,19287.450237,6.383886,0.473934
2,Bexley,26.685649,False,Bexley,63.082413,8.659319e+06,15.882363,19696.109589,6.506849,0.000000
3,Brent,46.512597,False,Brent,64.076461,6.242091e+06,25.199855,12034.705202,4.161850,5.780347
4,Bromley,30.871809,False,Bromley,63.344279,1.858615e+07,13.969447,21707.984772,7.106599,0.507614


In [18]:
master.columns

Index(['borough_name', 'no2_baseline_mean', 'no2_imputed',
       'no2_source_borough', 'lden_mean', 'green_area_m2', 'imd_score_mean',
       'imd_rank_mean', 'imd_decile_mean', 'pct_most_deprived'],
      dtype='object')

In [34]:
master.to_parquet(f"{DATA_DIR}/master_borough.parquet")

job = bq_client.load_table_from_dataframe(
    master,
    BQ_TABLE_MAIN,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
)

job.result()

print(f"Reloaded corrected master table to BigQuery: {BQ_TABLE_MAIN}")

Reloaded corrected master table to BigQuery: env-health-london-2026.env_health.master_borough


In [20]:
import os

"master_borough.parquet" in os.listdir(DATA_DIR)

True

In [21]:
from google.oauth2 import service_account
from google.cloud import bigquery

creds = service_account.Credentials.from_service_account_file(KEY_PATH)

bq_client = bigquery.Client(
    credentials=creds,
    project=GCP_PROJECT
)

job = bq_client.load_table_from_dataframe(
    master,
    BQ_TABLE_MAIN,
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    )
)

job.result()

print(f"Loaded master table to BigQuery: {BQ_TABLE_MAIN}")

/opt/anaconda3/envs/envhealth/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Loaded master table to BigQuery: env-health-london-2026.env_health.master_borough


In [35]:
query = f"""
SELECT borough_name, pct_most_deprived, imd_score_mean
FROM `{BQ_TABLE_MAIN}`
ORDER BY pct_most_deprived DESC
LIMIT 10
"""

bq_client.query(query).to_dataframe()

,borough_name,pct_most_deprived,imd_score_mean
0,Hackney,11.111111,32.868201
1,Haringey,9.655172,27.587200
2,Kensington and Chelsea,8.737864,22.019301
3,Brent,5.780347,25.199855
4,Enfield,5.464481,25.403448
5,Islington,4.878049,27.706041
6,Barking and Dagenham,3.636364,32.883473
7,Southwark,3.012048,26.041289
8,Lewisham,2.958580,26.869361
9,Newham,2.439024,29.771079


In [23]:
query = f"""
SELECT COUNT(*) AS row_count
FROM `{BQ_TABLE_MAIN}`
"""

bq_client.query(query).to_dataframe()

,row_count
0,33


In [24]:
master["no2_imputed"].value_counts()

no2_imputed
False    29
True      4
Name: count, dtype: int64

In [25]:
query = f"""
SELECT no2_imputed, COUNT(*) AS count
FROM `{BQ_TABLE_MAIN}`
GROUP BY no2_imputed
"""

bq_client.query(query).to_dataframe()

,no2_imputed,count
0,False,29
1,True,4


In [ ]:
# check

In [36]:
query = f"""
SELECT *
FROM `{BQ_TABLE_MAIN}`
LIMIT 10
"""

df_check = bq_client.query(query).to_dataframe()

df_check

,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough,lden_mean,green_area_m2,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
0,Bexley,26.685649,False,Bexley,63.082413,8.659319e+06,15.882363,19696.109589,6.506849,0.0
1,Camden,53.645008,False,Camden,64.701372,6.431268e+06,19.981053,16247.744361,5.406015,0.0
2,City of London,59.732950,False,City of London,64.333843,5.145284e+04,14.805000,20275.166667,6.666667,0.0
3,Harrow,30.330853,False,Harrow,63.580240,6.029958e+06,14.860883,19977.000000,6.554745,0.0
4,Hillingdon,37.919454,False,Hillingdon,61.948496,1.246287e+07,17.839329,17705.385093,5.906832,0.0
5,Kingston upon Thames,44.309768,False,Kingston upon Thames,63.607574,1.703662e+07,11.303245,23519.428571,7.622449,0.0
6,Lambeth,53.645176,False,Lambeth,63.611480,3.866588e+06,25.797247,11294.589888,3.949438,0.0
7,Merton,49.373426,False,Merton,63.614312,1.354176e+07,14.347331,20967.604839,6.887097,0.0
8,Redbridge,33.796433,False,Redbridge,62.988291,1.700287e+07,16.907851,18068.701863,5.987578,0.0
9,Richmond upon Thames,28.230027,False,Richmond upon Thames,62.908892,2.858851e+07,9.423122,25694.460870,8.321739,0.0


In [37]:
query = f"""
SELECT
  COUNT(*) AS total_boroughs,
  SUM(CASE WHEN pct_most_deprived > 0 THEN 1 ELSE 0 END) AS boroughs_above_zero,
  SUM(CASE WHEN pct_most_deprived = 0 THEN 1 ELSE 0 END) AS boroughs_equal_zero
FROM `{BQ_TABLE_MAIN}`
"""

bq_client.query(query).to_dataframe()

,total_boroughs,boroughs_above_zero,boroughs_equal_zero
0,33,22,11


In [38]:
query = f"""
SELECT *
FROM `{BQ_TABLE_MAIN}`
ORDER BY pct_most_deprived DESC
LIMIT 15
"""

df_check = bq_client.query(query).to_dataframe()
df_check

,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough,lden_mean,green_area_m2,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
0,Hackney,48.655825,False,Hackney,63.962610,3.994272e+06,32.868201,7420.611111,2.743056,11.111111
1,Haringey,28.675485,False,Haringey,64.065944,9.085837e+06,27.587200,11291.917241,3.903448,9.655172
2,Kensington and Chelsea,28.186041,False,Kensington and Chelsea,65.275871,2.515418e+06,22.019301,15419.339806,5.213592,8.737864
3,Brent,46.512597,False,Brent,64.076461,6.242091e+06,25.199855,12034.705202,4.161850,5.780347
4,Enfield,31.360600,False,Enfield,63.013682,1.248222e+07,25.403448,12698.519126,4.382514,5.464481
5,Islington,34.672520,False,Islington,64.512417,1.100483e+06,27.706041,10296.512195,3.642276,4.878049
6,Barking and Dagenham,23.116494,False,Barking and Dagenham,62.939922,6.590169e+06,32.883473,7244.500000,2.681818,3.636364
7,Southwark,35.653926,False,Southwark,64.135170,4.479264e+06,26.041289,11546.481928,4.024096,3.012048
8,Lewisham,37.932047,False,Lewisham,64.038411,6.524244e+06,26.869361,10781.378698,3.763314,2.958580
9,Newham,28.633707,False,Newham,63.416212,6.049061e+06,29.771079,8650.018293,3.091463,2.439024


In [39]:
query = f"""
SELECT
  MIN(pct_most_deprived) AS min_pct_most_deprived,
  MAX(pct_most_deprived) AS max_pct_most_deprived,
  AVG(pct_most_deprived) AS avg_pct_most_deprived
FROM `{BQ_TABLE_MAIN}`
"""

bq_client.query(query).to_dataframe()

,min_pct_most_deprived,max_pct_most_deprived,avg_pct_most_deprived
0,0.0,11.111111,2.197838
